In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
##PARAMETRIZAR ADLS Y CATALOGO A PROD
dbutils.widgets.text("PRM_catalogo","catalogo_desa_intEcommerce")
dbutils.widgets.text("PRM_nameStorage","adlsecompag")
#####################################
dbutils.widgets.text("PRM_container","raw-insumos")
dbutils.widgets.text("PRM_esquema", "bronze")

In [0]:
PRM_container = dbutils.widgets.get("PRM_container")
PRM_nameStorage = dbutils.widgets.get("PRM_nameStorage")
PRM_catalogo = dbutils.widgets.get("PRM_catalogo")
PRM_esquema = dbutils.widgets.get("PRM_esquema")


ruta = f"abfss://{PRM_container}@{PRM_nameStorage}.dfs.core.windows.net/ecommerce_data.csv"

In [0]:
df_interaccionecom = df_raw_tipointeraccion = spark.read \
                        .format("csv") \
                        .option("sep", ";") \
                        .option("header", "true") \
                        .option("inferSchema", "true") \
                        .load(ruta)

df_interaccionecom.display()                       

In [0]:
df_interaccionecom_final = df_interaccionecom.select(\
                            col("Interaction_id_system").alias("Interaction_id_system"),
                            col("Interaction_date").alias("Interaction_date"),
                            col("User_id_system").alias("User_id_system"),
                            col("Product_id").alias("Product_id"),
                            col("TypeInt_id").alias("TypeInt_id"),
                            col("Event").alias("Event"),
                            col("Location").alias("Location"),
                            col("Quantity").alias("Quantity"),
                            col("Product_rating").cast(DecimalType(10, 1))         
).withColumn("Fecha_proceso",to_timestamp(date_format(current_timestamp(), "yyyy-MM-dd")))


In [0]:
df_interaccionecom_final.write \
    .mode("overwrite") \
    .saveAsTable(f"{PRM_catalogo}.{PRM_esquema}.ecommerce_data")